In [12]:
# !pip install transformers

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# !pip install "transformers[torch]"

In [15]:
import pandas as pd
from transformers import T5Tokenizer, Trainer,TrainingArguments,T5ForConditionalGeneration

In [16]:
train_data = pd.read_csv("/samsum-train.csv")
validation_data = pd.read_csv("/samsum-validation.csv")

In [17]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [18]:
print(train_data.shape)
print(validation_data.shape)

(14732, 3)
(818, 3)


In [19]:
# Checking our Data
train_data['dialogue'][0]  # some nextline words(\n) and symbols (:-)are present in the text !

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [20]:
#Random Sampling
train_data = train_data.sample(n = 4000 , random_state = 42).reset_index(drop = True)
Valid_data = validation_data.sample(n = 500 , random_state = 42).reset_index(drop = True)

In [21]:
print(train_data.shape)
print(Valid_data.shape)

(4000, 3)
(500, 3)


### ***Data PreProcessing***

In [22]:
import re # regex

def clean_data(text):
  text = re.sub(r"\r\n"," ",text) # to remove next lines etc..
  text = re.sub(r"\s+"," ",text) # spaces
  text = re.sub(r"<.*?>"," ",text)# to remove HTML tags <p> ,<t1>
  text.strip().lower()

  return text

In [23]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

Valid_data['dialogue'] = Valid_data['dialogue'].apply(clean_data)
Valid_data['summary'] = Valid_data['summary'].apply(clean_data)

### ***Tokenization***

In [24]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [25]:
# raw data = > Tokenized input for fine-tuning/training

# Tokenize -> 1. padding 2.Max-length 3. truncate

def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding ="max_length",max_length = 512,truncation=  True)
    targets = tokenizer(data["summary"],padding ="max_length",max_length = 128,truncation=  True)

    inputs["labels"] = targets["input_ids"] # token_ids = > add to input as lables
    return inputs


  # inputs_id -> dialogue ->token_ids

  # 1 => EOS, 0=> Attention Mask
  # attention_mask(says about padding),
  # lables -targets =>Summary->token_ids(input_ids)

In [26]:
train_dataset = train_data.apply(tokenize,axis = 1).tolist()
valid_dataset = Valid_data.apply(tokenize,axis = 1).tolist()

In [27]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [28]:
print(len(train_dataset[0]["input_ids"]))
print(len(train_dataset[0]["labels"]))


512
128


# ***Working with the Model***

In [29]:
# NLP => Generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [30]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("device :",device)

model.to(device)

CUDA available: True
Device: Tesla T4
device : cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [31]:
## Training Arguments
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 8,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps= 500
    # o => lr default


)

In [32]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset
)

## **Train the Model**

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.224889,0.455488
2,0.475845,0.428484
3,0.447714,0.419544
4,0.431742,0.415907
5,0.421409,0.413200
6,0.414900,0.410650
7,0.410171,0.410532
8,0.406957,0.410284


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4000, training_loss=0.9042034530639649, metrics={'train_runtime': 1561.5467, 'train_samples_per_second': 20.493, 'train_steps_per_second': 2.562, 'total_flos': 4330937647104000.0, 'train_loss': 0.9042034530639649, 'epoch': 8.0})

In [53]:

trainer.save_model("/content/drive/MyDrive/TextSummarizer/final_model")
tokenizer.save_pretrained("/content/drive/MyDrive/TextSummarizer/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
!ls /content/drive/MyDrive/TextSummarizer/final_model

config.json		model.safetensors      tokenizer.json
generation_config.json	tokenizer_config.json  training_args.bin


In [56]:
!ls /content/drive/MyDrive/TextSummarizer/final_model/

config.json		model.safetensors      tokenizer.json
generation_config.json	tokenizer_config.json  training_args.bin


In [57]:
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/TextSummarizer/final_model")
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/TextSummarizer/final_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

***Testing the Summarizer Logic***

In [58]:
def summarize_dialogue(dialogue):

  dialogue = clean_data(dialogue) # clean the dialogue

  #tokenize
  inputs = tokenizer(
      dialogue,
      max_length = 512,
      padding = "max_length",
      truncation = True,
      return_tensors = "pt" #pt => # pytorch tensors
  )


  #generate the summary => token_ids
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 128,
      num_beams = 4,
      early_stopping = True

  )
  # token_ids => text ---->Decoding

  summary = tokenizer.decode(
      targets[0],
      skip_special_tokens = True,
      clean_up_tokenization_spaces = True
  )

  return summary

In [59]:
test_dialogue = """
Emma: Hey everyone! Are we still planning the weekend trip?
Liam: Definitely. I already checked the weather. It should be sunny on Saturday and cloudy on Sunday.
Sophia: Great! Where are we going finally? We had three options.
Emma: I think the hill station is the best choice. It's only a three-hour drive.
Noah: I agree. The beach will probably be too crowded.
Olivia: Same here. The hill station sounds relaxing.
Liam: I can drive. My car has space for five people.
Emma: Perfect. We are six though.
Sophia: My brother said we could borrow his SUV if needed.
Noah: That would solve the problem.
Olivia: What time should we leave?
Emma: Around 6:30 AM so we can avoid traffic.
Liam: Works for me.
Sophia: Same.
Noah: Me too.
Olivia: I'll bring snacks for everyone.
Emma: Thanks! I'll handle breakfast sandwiches.
Liam: I'll carry water bottles and juice.
Sophia: I'll make a playlist for the road trip.
Noah: I'll book the hotel tonight.
Emma: Please make sure it has parking.
Noah: Already checked. Free parking and breakfast included.
Olivia: Nice! How much is the hotel?
Noah: Around ₹12,000 for two rooms.
Emma: That means ₹2,000 per person.
Liam: That's reasonable.
Sophia: Should we also visit the waterfall nearby?
Emma: Yes! It's just 20 minutes from the hotel.
Noah: We can go there on Saturday afternoon.
Olivia: What about dinner?
Liam: I found a restaurant with great reviews.
Sophia: Can you share the location?
Liam: Sure, sending it now.
Emma: Received it.
Noah: Looks amazing.
Olivia: Do we need to carry jackets?
Liam: Yes. The temperature drops to around 15°C at night.
Sophia: Thanks for reminding us.
Emma: Don't forget your ID cards. The hotel requires them.
Noah: Good point.
Olivia: I'll also bring a first-aid kit.
Liam: Smart idea.
Sophia: Should we split the fuel cost too?
Emma: Yes, we'll divide it equally after the trip.
Noah: Fine with me.
Olivia: Same here.
Emma: Awesome! I'll create a checklist and share it tonight.
Liam: Looking forward to the trip.
Sophia: Can't wait!
Noah: This is going to be fun.
Olivia: See you all Saturday morning!
"""

summary = summarize_dialogue(test_dialogue)
print(summary)

Liam and Olivia are planning the weekend trip. The hill station is only a three-hour drive. Olivia will bring snacks for everyone. Olivia will also bring a first-aid kit.


In [46]:
print(type(model))

<class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>
